# 3 · Evaluate the fine-tuned MedCAT model on an annotated test set

Runs the full model from notebook 2 (`..._medroberta_<version>.zip`) on the test notes and compares its output with the MedCATtrainer **test export**:

1. **Concept recognition and linking (NER+L):** precision, recall and F1, overall and per SNOMED CT concept, with
   - *strict* matching: exactly the same span and concept,
   - *lenient* matching: overlapping span, same concept.
2. **Status (Affirmed / Negated), end to end:** for mentions the model found (lenient match), is its Status label correct?
3. **Status on the annotated spans:** the MetaCAT classifier alone, run on all annotated mentions (independent of NER errors).

The test notes must not have been used for training: keep them in a separate MedCATtrainer project with *Train Model On Submit* off, and never list the test export in notebook 2.

## Configuration

In [ ]:
from pathlib import Path

MODEL_PACK = "medcat_data/models/snomed_symptoms_en_nl_medroberta_v1.zip"   # full pack from notebook 2
TEST_EXPORTS = [
    "medcat_data/exports/round1_test_export.json",
]
STATUS_TASK = "Status"
APPLY_PROJECT_CUI_FILTER = True   # ignore predictions outside a project's CUI filter, as annotators never saw them

RESULTS_DIR = Path("medcat_data/evaluation") / Path(MODEL_PACK).stem
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## MedRoBERTa.nl patch
Needed before loading a model pack with the MedRoBERTa.nl MetaCAT (see notebook 2).

In [ ]:
from transformers import AutoModel, AutoTokenizer
import medcat.components.addons.meta_cat.models as _mc_models
import medcat.components.addons.meta_cat.mctokenizers.bert_tokenizer as _mc_bert_tok

_mc_models.BertModel = AutoModel
_mc_bert_tok.BertTokenizerFast = AutoTokenizer

## Load the model and the test annotations

In [ ]:
import json
import pandas as pd
from medcat.cat import CAT
from medcat.components.addons.meta_cat.meta_cat import MetaCATAddon

cat = CAT.load_model_pack(MODEL_PACK)
metacats = {a.mc.config.general.category_name: a.mc for a in cat.get_addons_of_type(MetaCATAddon)}
if STATUS_TASK not in metacats:
    raise ValueError(f"No MetaCAT {STATUS_TASK!r} in {MODEL_PACK}; found {list(metacats)}")


def meta_anns_of(ann):
    m = ann.get("meta_anns") or []
    return list(m.values()) if isinstance(m, dict) else list(m)


def is_gold(ann):
    """A human-validated, correct (or corrected) concept annotation."""
    return (ann.get("validated", True) and (ann.get("correct", False) or ann.get("alternative", False))
            and not ann.get("deleted", False) and not ann.get("killed", False) and not ann.get("irrelevant", False))


def project_cuis(project):
    cuis = project.get("cuis") or []
    if isinstance(cuis, str):
        cuis = [c.strip() for c in cuis.split(",")]
    return {c for c in cuis if c}


test_projects = []
for path in TEST_EXPORTS:
    with open(path, encoding="utf-8") as f:
        test_projects.extend(json.load(f)["projects"])
print(f"{len(test_projects)} project(s), {sum(len(p['documents']) for p in test_projects)} documents")

## Run the model on the test notes

In [ ]:
gold_rows, pred_rows = [], []
for p in test_projects:
    cui_filter = project_cuis(p) if APPLY_PROJECT_CUI_FILTER else set()
    for d in p["documents"]:
        doc_id = f"{p['name']}::{d.get('name', d.get('id'))}"
        for a in d.get("annotations", []):
            if is_gold(a):
                status = next((m["value"] for m in meta_anns_of(a) if m.get("name") == STATUS_TASK), None)
                gold_rows.append({"doc": doc_id, "start": a["start"], "end": a["end"], "cui": str(a["cui"]),
                                  "text": d["text"][a["start"]:a["end"]], "status": status})
        for e in cat.get_entities(d["text"])["entities"].values():
            if cui_filter and e["cui"] not in cui_filter:
                continue
            s = (e.get("meta_anns") or {}).get(STATUS_TASK, {})
            pred_rows.append({"doc": doc_id, "start": e["start"], "end": e["end"], "cui": e["cui"],
                              "text": e["source_value"], "acc": e["acc"],
                              "status": s.get("value"), "status_confidence": s.get("confidence")})

gold_df = pd.DataFrame(gold_rows, columns=["doc", "start", "end", "cui", "text", "status"])
pred_df = pd.DataFrame(pred_rows, columns=["doc", "start", "end", "cui", "text", "acc", "status", "status_confidence"])
print(f"{len(gold_df)} gold mentions, {len(pred_df)} predicted mentions")

## Match predictions to gold annotations
One-to-one matching per document: first exact matches, then (lenient) overlapping spans with the same concept.

In [ ]:
def match(gold_doc, pred_doc):
    """Return [(gold_idx, pred_idx, 'strict'|'lenient')] for one document."""
    pairs, used_g, used_p = [], set(), set()
    for gi, g in gold_doc.iterrows():
        for pi, p in pred_doc.iterrows():
            if pi not in used_p and (g.start, g.end, g.cui) == (p.start, p.end, p.cui):
                pairs.append((gi, pi, "strict")); used_g.add(gi); used_p.add(pi); break
    for gi, g in gold_doc.iterrows():
        if gi in used_g:
            continue
        for pi, p in pred_doc.iterrows():
            if pi not in used_p and g.cui == p.cui and g.start < p.end and p.start < g.end:
                pairs.append((gi, pi, "lenient")); used_g.add(gi); used_p.add(pi); break
    return pairs


pairs = []
for doc in set(gold_df["doc"]) | set(pred_df["doc"]):
    pairs += match(gold_df[gold_df["doc"] == doc], pred_df[pred_df["doc"] == doc])

gold_df["match"], pred_df["match"] = "FN", "FP"
gold_df["pred_status"], pred_df["gold_status"] = None, None
for gi, pi, kind in pairs:
    gold_df.loc[gi, "match"] = pred_df.loc[pi, "match"] = kind
    gold_df.loc[gi, "pred_status"] = pred_df.loc[pi, "status"]
    pred_df.loc[pi, "gold_status"] = gold_df.loc[gi, "status"]

## 1 · Concept recognition and linking (NER+L)

In [ ]:
def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0, "tp": tp, "fp": fp, "fn": fn}


n_strict = int((gold_df["match"] == "strict").sum())
n_lenient = n_strict + int((gold_df["match"] == "lenient").sum())
ner = pd.DataFrame({
    "strict": prf(n_strict, len(pred_df) - n_strict, len(gold_df) - n_strict),
    "lenient": prf(n_lenient, len(pred_df) - n_lenient, len(gold_df) - n_lenient),
}).T
print(ner.round(3).to_string())


def concept_name(cui):
    info = cat.cdb.cui2info.get(cui)
    return info["preferred_name"] if info else "(not in CDB)"


rows = []
for cui in sorted(set(gold_df["cui"]) | set(pred_df["cui"])):
    g, p = gold_df[gold_df["cui"] == cui], pred_df[pred_df["cui"] == cui]
    tp = int(g["match"].isin(["strict", "lenient"]).sum())
    rows.append({"cui": cui, "name": concept_name(cui), "gold": len(g), "pred": len(p),
                 **prf(tp, len(p) - tp, len(g) - tp)})
per_concept = pd.DataFrame(rows).sort_values("gold", ascending=False)
print("\nPer concept (lenient):")
print(per_concept.round(3).to_string(index=False))

## 2 · Status, end to end
Only for gold mentions that the model found (lenient match) and that have a Status label.

In [ ]:
from sklearn.metrics import classification_report

found = gold_df[gold_df["match"].isin(["strict", "lenient"]) & gold_df["status"].notna()]
if len(found):
    print(classification_report(found["status"], found["pred_status"].fillna("(none)"), digits=3, zero_division=0))
    print(pd.crosstab(found["status"], found["pred_status"].fillna("(none)"),
                      rownames=["gold"], colnames=["predicted"]).to_string())
else:
    print("No matched mentions with a Status label.")

## 3 · Status on the annotated spans (MetaCAT only)

In [ ]:
mc = metacats[STATUS_TASK]
metacat_results = {}
for path in TEST_EXPORTS:
    res = mc.eval(path)
    metacat_results[path] = {k: res[k] for k in ("precision", "recall", "f1")}
    print(f"{path}: weighted precision={res['precision']:.3f} recall={res['recall']:.3f} f1={res['f1']:.3f}")
    print(res["confusion matrix"].to_string(), "\n")

## Save results and inspect errors

In [ ]:
pred_df.to_csv(RESULTS_DIR / "predictions.csv", index=False)
gold_df.to_csv(RESULTS_DIR / "gold.csv", index=False)
per_concept.to_csv(RESULTS_DIR / "per_concept.csv", index=False)
summary = {
    "model_pack": MODEL_PACK, "test_exports": TEST_EXPORTS,
    "ner": ner.to_dict(orient="index"),
    "status_end_to_end": (classification_report(found["status"], found["pred_status"].fillna("(none)"),
                                                output_dict=True, zero_division=0) if len(found) else None),
    "status_on_gold_spans": metacat_results,
}
with open(RESULTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=float)
print("Results written to", RESULTS_DIR)

pd.set_option("display.max_colwidth", 80)
print("\nMissed (FN):")
print(gold_df[gold_df["match"] == "FN"][["doc", "text", "cui", "status"]].head(20).to_string(index=False))
print("\nSpurious (FP):")
print(pred_df[pred_df["match"] == "FP"][["doc", "text", "cui", "acc"]].head(20).to_string(index=False))
print("\nWrong Status:")
wrong = found[found["status"] != found["pred_status"]]
print(wrong[["doc", "text", "status", "pred_status"]].head(20).to_string(index=False))